In [ ]:
import pandas as pd
from tabpfn import TabPFNClassifier, TabPFNRegressor
from tabpfn_extensions.embedding import TabPFNEmbedding
import numpy as np
from dotenv import load_dotenv
from sklearn.model_selection import StratifiedShuffleSplit
load_dotenv(".env")

In [2]:
from benchmark_datasets import OpenMLBenchmark

In [4]:
bench = OpenMLBenchmark()
print("Benchmark suite (20 datasets):")
print(bench.list().to_string(index=False))

ds = bench.load("lymph")
print(f"\nLoaded {ds.name!r}: X={ds.X.shape} ({ds.X.dtype}), y={ds.y.shape}")
print(f"classes: {ds.n_classes}, feature count: {len(ds.feature_names)}")

Benchmark suite (20 datasets):
                  name  data_id           task  n_rows                                                note
           tic-tac-toe       50 classification     958             XOR-like win patterns, pure interaction
      monks-problems-2      334 classification     601                       synthetic XOR / parity target
                 sonar       40 classification     208      60 correlated sonar bands, non-linear boundary
            ionosphere       59 classification     351                            radar signal, non-linear
               vehicle       54 classification     846                         4-class silhouette geometry
                  wdbc     1510 classification     569      breast cancer, non-linear feature interactions
              diabetes       37 classification     768           Pima diabetes, classic non-linear medical
                  ilpd     1480 classification     583            Indian liver patient, non-linear medical
      

In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neural_network import MLPRegressor
from aug_dataset import aug_dataset

In [7]:
from sklearn.metrics import accuracy_score
sss = StratifiedShuffleSplit(1,train_size=64)
train_index, test_index = next(sss.split(ds.X, ds.y))
X_train = ds.X[train_index]
y_train = ds.y[train_index]
X_test = ds.X[test_index]
y_test = ds.y[test_index]

print('Num labeled samples:', len(X_train))

clf = TabPFNClassifier()
clf.fit(X_train, y_train)
base_pfn_pred = clf.predict(X_test)
print('Base tabPFN acc: ', accuracy_score(y_test,base_pfn_pred))

X_aug = aug_dataset(3, X_train, 0.8, 1, 0.5) 

X_aug_conf_indx = abs(0.5-clf.predict_proba(X_aug)[:, 0])>0.45
X_aug_conf = X_aug[X_aug_conf_indx]
print('Num augmented samples:', len(X_aug))
print('Num confident augmented samples:', len(X_aug_conf))

y_aug = clf.predict(X_aug)
y_aug_conf = clf.predict(X_aug_conf)

y_sl_aug = clf.predict_proba(X_aug)[:,1]

print('done')

Num labeled samples: 64
Base tabPFN acc:  0.8095238095238095
Num augmented samples: 256
Num confident augmented samples: 228
done


In [8]:
results = []

for n_estim in [1,2,3,5,10,100]:
    rfr = RandomForestClassifier(n_estimators=n_estim)
    rfr.fit(X_train, y_train)
    base_rfr_pred = rfr.predict(X_test)
    print(F'RFR (n_estim={n_estim}) base acc: ', accuracy_score(y_test,base_rfr_pred))
    

    rfr_aug = RandomForestClassifier(n_estimators=n_estim)
    rfr_aug.fit(X_aug, y_aug)
    rfr_aug_pred = rfr_aug.predict(X_test)
    print(F'RFR (n_estim={n_estim}) aug acc: ', accuracy_score(y_test,rfr_aug_pred))

    rfr_aug_conf = RandomForestClassifier(n_estimators=n_estim)
    rfr_aug_conf.fit(X_aug_conf, y_aug_conf)
    rfr_aug_conf_pred = rfr_aug_conf.predict(X_test)
    print(F'RFR (n_estim={n_estim}) aug conf acc: ', accuracy_score(y_test,rfr_aug_conf_pred))

    results.append({'name': f'RFR n_estim={n_estim}', 
                    'base': accuracy_score(y_test,base_rfr_pred),
                    'aug': accuracy_score(y_test, rfr_aug_pred),
                    'aug confident': accuracy_score(y_test, rfr_aug_conf_pred),
                    'solf-labels aug': None})

for hid_size in [(), (8,),(32,),(128,), (16,16,), (32,32), (16,16,16,), (67,67,67,)]:
    mlp = MLPClassifier(hidden_layer_sizes=hid_size, max_iter=5000)
    mlp.fit(X_train, y_train)
    base_mlp_pred = mlp.predict(X_test)
    print(f'MLP (hidden_l={hid_size}) base acc: ', accuracy_score(y_test,base_mlp_pred))

    mlp_aug = MLPClassifier(hidden_layer_sizes=hid_size, max_iter=5000)
    mlp_aug.fit(X_aug, y_aug)
    mlp_aug_pred= mlp_aug.predict(X_test)
    print(f'MLP (hidden_l={hid_size}) aug acc: ', accuracy_score(y_test,mlp_aug_pred))

    mlp_aug_conf = MLPClassifier(hidden_layer_sizes=hid_size, max_iter=5000)
    mlp_aug_conf.fit(X_aug_conf, y_aug_conf)
    mlp_aug_conf_pred= mlp_aug_conf.predict(X_test)
    print(f'MLP (hidden_l={hid_size}) aug conf acc: ', accuracy_score(y_test,mlp_aug_conf_pred))

    mlp_sl_aug = MLPRegressor(hidden_layer_sizes=hid_size, max_iter=5000)
    mlp_sl_aug.fit(X_aug, y_sl_aug)
    mlp_sl_aug_pred = mlp_sl_aug.predict(X_test) > 0.5
    print(f'MLP (hidden_l={hid_size}) soft-labels aug acc: ', accuracy_score(y_test,mlp_sl_aug_pred))

    results.append({'name': f'MLP hidden_l={hid_size}', 
                'base': accuracy_score(y_test,base_mlp_pred),
                'aug': accuracy_score(y_test, mlp_aug_pred),
                'aug confident': accuracy_score(y_test, mlp_aug_conf_pred),
                'solf-labels aug': accuracy_score(y_test, mlp_sl_aug_pred)})

results_df = pd.DataFrame(results)
results_df

RFR (n_estim=1) base acc:  0.7023809523809523
RFR (n_estim=1) aug acc:  0.7857142857142857
RFR (n_estim=1) aug conf acc:  0.7380952380952381
RFR (n_estim=2) base acc:  0.7261904761904762
RFR (n_estim=2) aug acc:  0.6547619047619048
RFR (n_estim=2) aug conf acc:  0.7857142857142857
RFR (n_estim=3) base acc:  0.7857142857142857
RFR (n_estim=3) aug acc:  0.8333333333333334
RFR (n_estim=3) aug conf acc:  0.7619047619047619
RFR (n_estim=5) base acc:  0.7976190476190477
RFR (n_estim=5) aug acc:  0.7619047619047619
RFR (n_estim=5) aug conf acc:  0.8095238095238095
RFR (n_estim=10) base acc:  0.8452380952380952
RFR (n_estim=10) aug acc:  0.7857142857142857
RFR (n_estim=10) aug conf acc:  0.7976190476190477
RFR (n_estim=100) base acc:  0.7857142857142857
RFR (n_estim=100) aug acc:  0.7619047619047619
RFR (n_estim=100) aug conf acc:  0.7857142857142857
MLP (hidden_l=()) base acc:  0.8095238095238095
MLP (hidden_l=()) aug acc:  0.7976190476190477
MLP (hidden_l=()) aug conf acc:  0.785714285714285

,name,base,aug,aug confident,solf-labels aug
0,RFR n_estim=1,0.702381,0.785714,0.738095,NaN
1,RFR n_estim=2,0.726190,0.654762,0.785714,NaN
2,RFR n_estim=3,0.785714,0.833333,0.761905,NaN
3,RFR n_estim=5,0.797619,0.761905,0.809524,NaN
4,RFR n_estim=10,0.845238,0.785714,0.797619,NaN
5,RFR n_estim=100,0.785714,0.761905,0.785714,NaN
6,MLP hidden_l=(),0.809524,0.797619,0.785714,0.333333
7,"MLP hidden_l=(8,)",0.761905,0.821429,0.833333,0.333333
8,"MLP hidden_l=(32,)",0.797619,0.809524,0.773810,0.297619
9,"MLP hidden_l=(128,)",0.797619,0.797619,0.773810,0.297619
